<a href="https://colab.research.google.com/github/Maximi652/efficient-slm-architectures/blob/main/LotteryTicketSLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets bitsandbytes
from google.colab import drive
drive.mount('/content/drive')

import torch, gc
import torch.nn.utils.prune as prune
import copy
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
from bitsandbytes.optim import AdamW8bit

# ==== Konfigurierbare Konstanten ====
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/12B_trainingdata.json"
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B"
OUTPUT_PATH = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B-LTH_v3"
TRAIN_OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B-LTH_train_v2"
LOG_DIR = f"{TRAIN_OUTPUT_DIR}/logs"

# Trainingsdaten
with open("/content/drive/MyDrive/Colab Notebooks/12B_trainingdata.json", "r", encoding="utf-8") as f:
    data = json.load(f)["questions"]

rows = []
for entry in data:
    qtext = entry["body"].strip()
    qtype = entry["type"].lower()

    # ideal_answer
    ideal_ans = entry.get("ideal_answer", [""])[0].strip()
    if ideal_ans:
        if qtype == "yesno":
            user_msg = f"Question: {qtext}\nProvide one-sentence ideal answer in English starting with 'Yes,' or 'No,'."
        else:
            user_msg = f"Question: {qtext}\nProvide an ideal answer in English (one paragraph, maximum 200 words, full sentences)."
        prompt = (
            "<|im_start|>system\n/no_think\n<|im_end|>\n"
            "<|im_start|>user\n" + user_msg + "\n<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        rows.append({"text": prompt + ideal_ans})

    # exact_answer
    exact = entry.get("exact_answer", [])
    flat_exact = [item[0] if isinstance(item, list) and item else item for item in exact]
    exact_ans = ", ".join(flat_exact).strip()
    if exact_ans:
        if qtype == "yesno":
            user_msg = f"Question: {qtext}\nAnswer only 'yes' or 'no', in English, no extras."
        elif qtype == "factoid":
            user_msg = f"Question: {qtext}\nProvide up to 5 keywords, comma-separated, in English, no commentary."
        elif qtype == "list":
            user_msg = f"Question: {qtext}\nProvide a comma-separated list of relevant items, in English, no filler words."
        else:
            user_msg = f"Question: {qtext}\nProvide a brief answer in English."
        prompt = (
            "<|im_start|>system\n/no_think\n<|im_end|>\n"
            "<|im_start|>user\n" + user_msg + "\n<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        rows.append({"text": prompt + exact_ans})

# Training nur auf 100 Fragen
rows = rows[:100]

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = Dataset.from_list(rows)
tokenized_ds = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

N_PRUNE_ITER = 3
SPARSITY_TARGET = 0.25
PRUNE_EACH = 1 - (1 - SPARSITY_TARGET) ** (1 / N_PRUNE_ITER)

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/Qwen3-4B-LTH_train_v2",
    num_train_epochs=1,  # Wenige Epochen pro Iteration reichen für LTH-Test!
    per_device_train_batch_size=1,
    # save_steps=20,
    # save_total_limit=1,
    gradient_checkpointing=False,
    logging_steps=5,
    logging_dir="/content/drive/MyDrive/Colab Notebooks/Qwen3-4B-LTH_train_v2/logs",
    # fp16=True,
    report_to=[],
    optim="adamw_8bit"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH,
                                             torch_dtype=torch.float16,
                                             trust_remote_code=True
                                             ).to(device)

model_cpu = model.cpu().to(torch.float32)
global_mask = {}
for name, module in model_cpu.named_modules():
    if isinstance(module, torch.nn.Linear):
        global_mask[name] = torch.ones_like(module.weight.data, dtype=torch.bool)
init_state = copy.deepcopy(model_cpu.state_dict())

del model_cpu  # Speicher freigeben

def train_model(model, tokenized_ds):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds,
        # data_collator=data_collator,
        # optimizers=(AdamW8bit(model.parameters(), lr=5e-5), None),
    )
    trainer.train()

    del trainer
    torch.cuda.empty_cache()
    gc.collect()

def prune_model(model, amount):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            if hasattr(module, 'weight') and module.weight.dtype in [torch.float32, torch.float16]:
                prune.l1_unstructured(module, name='weight', amount=amount)

def lth_rewind(model, init_state):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear) and hasattr(module, 'weight_orig'):
            mask = module.weight_mask
            key = f"{name}.weight"
            if key in init_state:
                with torch.no_grad():
                    module.weight_orig.data = (1-mask)*init_state[key].to(module.weight_orig.device) + mask*module.weight_orig.data

    torch.cuda.empty_cache()
    gc.collect()

def remove_all_pruning(model):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear) and hasattr(module, "weight_mask"):
            prune.remove(module, "weight")

def check_sparsity(model):
    total, zeros = 0, 0
    for p in model.parameters():
        total += p.numel()
        zeros += (p == 0).sum().item()
    print(f"Sparsity: {zeros/total*100:.2f}%")

print("Initial Sparsity:")
check_sparsity(model)

torch.cuda.empty_cache()
gc.collect()

for i in range(N_PRUNE_ITER):
    print(f"\n=== Iteration {i+1}/{N_PRUNE_ITER} | Prune {PRUNE_EACH*100:.2f}% ===")

    # 1. Training (auf GPU/float16)
    train_model(model, tokenized_ds)

    # 2. Pruning auf CPU/float32
    model_cpu = model.cpu().to(torch.float32)
    prune_model(model_cpu, PRUNE_EACH)

    # 4. Rewind (immer auf float32/CPU!)
    lth_rewind(model_cpu, init_state)
    remove_all_pruning(model_cpu)

    # 3. Sparsity prüfen (auf float32)
    check_sparsity(model_cpu)

    # 5. Modell zurück auf GPU/float16
    del model
    torch.cuda.empty_cache()
    gc.collect()
    model = model_cpu.to(device).to(torch.float16)
    del model_cpu
    gc.collect()

print("\nNach remove_pruning:")
check_sparsity(model)

model.save_pretrained(OUTPUT_PATH, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_PATH)

In [ ]:
print(torch.cuda.memory_summary())

In [ ]:
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"{name:<50} | {type(module).__name__:<20} | dtype: {module.weight.dtype}")


# Inferenz

In [ ]:
# Install und Imports
!pip install -q transformers datasets

import json
from datasets import Dataset
import torch
from torch import inference_mode
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# Pfade und Gerät
model_name = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B_LTH_25"
test_json_path = "/content/drive/MyDrive/Colab Notebooks/12b_golden_testdata.json"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Basis-Modell und LoRA-Adapter
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True
).to(device)

# Testdaten parsen und formatieren
def format_entry_chat(entry):

    qtext = entry["body"].strip()
    qtype = entry["type"].lower()
    # ctx = "\n".join(s["text"].strip() for s in list(entry["snippets"]))
    results = []

    # Ausformulierte Antwort
    ideal_ans = entry["ideal_answer"][0].strip()
    if ideal_ans:
        if qtype == "yesno":
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide one-sentence ideal answer in English starting with 'Yes,' or 'No,'."
            user_msg = f"Question: {qtext}\nProvide one-sentence ideal answer in English starting with 'Yes,' or 'No,'."
        else:
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide an ideal answer in English (one paragraph, max 200 words, full sentences)."
            user_msg = f"Question: {qtext}\nProvide an ideal answer in English (one paragraph, maximum 200 words, full sentences)."
        messages = [
            {"role": "system", "content": "/no_think"},
            {"role": "user", "content": user_msg},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
        # text = (
        # "<|im_start|>system\n"
        # "/no_think\n"
        # "<|im_end|>\n"
        # "<|im_start|>user\n"
        # f"{user_msg}\n"
        # "<|im_end|>\n"
        # "<|im_start|>assistant\n"
        # )
        results.append({"text": text})
        # print("Prompt:", repr(text))

    # Kurzantwort
    try:

        answer = entry["exact_answer"]

    except KeyError:

        answer = ""

    exact = answer
    flat_exact = [item[0] if isinstance(item, list) and item else item for item in exact]
    exact_ans = ", ".join(flat_exact).strip()
    if exact_ans:
        if qtype == "yesno":
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nAnswer only 'yes' or 'no', in English, no extras."
            user_msg = f"Question: {qtext}\nAnswer only 'yes' or 'no', in English, no extras."
        elif qtype == "factoid":
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide up to 5 keywords, comma-separated, in English, no commentary."
            user_msg = f"Question: {qtext}\nProvide up to 5 keywords, comma-separated, in English, no commentary."
        elif qtype == "list":
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide a comma-separated list of relevant items, in English, no filler words."
            user_msg = f"Question: {qtext}\nProvide a comma-separated list of relevant items, in English, no filler words."
        else:
            # user_msg = f"Question: {qtext}\nContext:\n{ctx}\nProvide a brief answer in English."
            user_msg = f"Question: {qtext}\nProvide a brief answer in English."
        messages = [
            {"role": "system", "content": "/no_think"},
            {"role": "user", "content": user_msg},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
        # text = (
        # "<|im_start|>system\n"
        # "/no_think\n"
        # "<|im_end|>\n"
        # "<|im_start|>user\n"
        # f"{user_msg}\n"
        # "<|im_end|>\n"
        # "<|im_start|>assistant\n"
        # )
        results.append({"text": text})
        # print("Prompt:", repr(text))

    return results

with open(test_json_path, "r", encoding="utf-8") as f:
    raw_test = json.load(f)["questions"]

# Flatten
formatted_test = []
for entry in raw_test:
    formatted_test.extend(format_entry_chat(entry))

def extract_qwen_answer(output_text):
    # Nach dem letzten </think> splitten
    if "</think>" in output_text:
        output_text = output_text.split("</think>")[-1]
    elif "assistant" in output_text:
        output_text = output_text.split("assistant", 1)[-1]
    # Entferne mögliche System-/User-/Leerzeilen vorne
    lines = output_text.strip().splitlines()
    answer_lines = [
        line for line in lines
        if not line.strip().lower() in {"system", "user", "assistant", "<think>", "</think>", ""}  # Nur echte Antwortzeilen
    ]
    return "\n".join(answer_lines).strip()

# Inference
def generate_answer(prompt_text, ):

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        padding=False        # gar nicht auf max length padden
    ).to(device)

    # Sampling-Parameter anpassen
    with inference_mode():
        output_ids = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id
        )

    # 3) Volltext decodieren & Prompt entfernen
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Output vergleichen
    # print("="*80)
    # print("PROMPT:")
    # print(repr(prompt_text))
    # print("OUTPUT:")
    # print(repr(output_text))
    # print("="*80)

    final_answer = extract_qwen_answer(output_text)

    return final_answer

# Loop über den Testdatensatz
results = []

for idx, ex in enumerate(formatted_test):
    prompt = ex["text"]
    pred = generate_answer(prompt)
    results.append({
        "index": idx,
        # "question": ex["question"],
        "prompt": prompt,
        "prediction": pred
    })

    print(pred)
    print("-" * 50)
    if idx % 20 == 0:
        print(f"Processed {idx}/{len(formatted_test)}")

# 8. Ergebnisse speichern oder auswerten
import json
with open("/content/drive/MyDrive/Colab Notebooks/test_predictions_LTH_25.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Inference abgeschlossen – Ergebnisse in test_predictions_LTH_25.json gespeichert.")